<a href="https://colab.research.google.com/github/rabeebanisalman/alexandria-quantum-hackathon-2026/blob/main/team12.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Track 1: Quantum Simulation of Photo-Induced Charge Dynamics ( From Ethylene to Photosynthesis )
## Task 1: Constructing the Ethylene Hamiltonian and Mapping to Qubits
### Ethylene Geometry at Equilibrium

```

 H       H
  \     /  
   C = C   
  /     \  
 H       H

```
C=C Bond Length $\approx 1.34Å$

C-H Bond Length $\approx 1.087Å$

H-C-H Bond Angle $\approx 117.3^{\circ}$

Carbons are relatively positioned at $ x=0, y=0 $ and aligned along z-axis where the distance of each from the origin is equal in value.

Carbon $ |d_z| = \frac{1.34}{2}Å = 0.67Å $

At $\pm0.67$ on the z-axis where the carbon atoms are located, bonds with the hydrogen atoms form. These bonds are tilted by an angle. We calculate the displacements on $y, z$ axes.

Hydrogen $|d_y| = 1.087\sin(\frac{117.3}{2}) = 0.928Å$

Hydrogen $|d_z| = 1.087\cos(\frac{117.3}{2}) + $ Carbon $|d_z| = 0.5655Å + 0.67Å = 1.2355Å$

Meanwhile there is no displacement on the x axis because it is planar.

Now we can finally construct the PySCF driver for Ethylene at equilibrium as follows:

In [ ]:
!pip install -q qiskit
!pip install -q qiskit-aer
!pip install -q qiskit-algorithms
!pip install -q qiskit-nature
!pip install -q qiskit-nature-pyscf
!pip install -q pyscf
!pip install -q scipy matplotlib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 76.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 MB 21.1 MB/s eta 0:00:00


In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver

eth_mol = PySCFDriver(
    atom="C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; C 0 0 -0.67; H 0 0.928 -1.2355; H 0 -0.928 -1.2355",
    basis='sto3g'
)

es_problem = eth_mol.run()

### Ethylene Geometry at a Twisted $90^{\circ}$ Angle ($CH_2$ Groups Perpendicular to Eachother)

```
  H H     
  |  \    
  C - C   
  |    \  
  H     H
```

Twisting one of the $CH_2$ groups $90^\circ$ isn't merely a change in the axes,  such alteration eliminates the spatial overlap which breaks the  $\pi$ bond into a relatively longer $\sigma$ bond. It also causes change in the C-H bond length and the bond angle because of the general change in the electronic structure.

The approximated new lengths and angle are as follows:

C-C Bond Length $\approx 1.46Å$

C-H Bond Length $\approx 1.08Å$

H-C-H Bond Angle $\approx 121^{\circ}$

Carbon $|d_z| = 1.46/2 = 0.73Å$

But now after the twist, hydrogens are on the x-axis instead of y where the new displacement on the x-axis is equal to the one on the y-axis at equilibrium, while remaining on the z-axis as well

Hydrogen $|d_x| = 0.928$

Hydrogen $|d_z| = 1.08\cos(\frac{121}{2}) +$ Carbon $ |d_z| = 0.53Å + 0.73Å = 1.26Å$



In [ ]:
eth_mol_twisted = PySCFDriver(
    atom="C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; "        # CH2 #1: unchanged
         "C 0 0 -0.67; H -0.928 0 -1.2355; H 0.928 0 -1.2355",      # CH2 #2: (x,y) -> (-y,x)
    basis='sto3g'
)

es_problem_twisted = eth_mol_twisted.run()

### Reducing to Active Space

2 electrons, 2 orbitals expandable to 2 electrons, 4 orbitals

In [ ]:
n_e = 2
n_orb = 2  # change to 4 for expansion

from qiskit_nature.second_q.transformers import ActiveSpaceTransformer

transformer = ActiveSpaceTransformer(num_electrons=n_e, num_spatial_orbitals=n_orb)

reduced_problem = transformer.transform(es_problem)
reduced_problem_twisted = transformer.transform(es_problem_twisted)


### Mapping to Qubits and Getting Hamiltonian for Ethylene in Equilibrium and Twisted Geometry

In [ ]:
from qiskit_nature.second_q.mappers import JordanWignerMapper

mapper = JordanWignerMapper()
h = mapper.map(reduced_problem.second_q_ops()[0])  # index of the main hamiltonian in returned tuple
h_twisted = mapper.map(reduced_problem_twisted.second_q_ops()[0])

print(h)
print(h_twisted)

SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'ZIII', 'ZIIZ', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'IZZI', 'ZIZI', 'ZZII'],
              coeffs=[-0.67810865+0.j,  0.07683157+0.j, -0.07802605+0.j,  0.08418098+0.j,
  0.07683157+0.j,  0.12685108+0.j, -0.07802605+0.j,  0.12720766+0.j,
  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,  0.04302668+0.j,
  0.12720766+0.j,  0.13086927+0.j,  0.08418098+0.j])
SparsePauliOp(['IIII', 'IIIZ', 'IIZI', 'IIZZ', 'IZII', 'IZIZ', 'YYII', 'YYIZ', 'XXII', 'XXIZ', 'ZIII', 'ZIIZ', 'IIYY', 'IZYY', 'IIXX', 'IZXX', 'YYYY', 'XXYY', 'YYXX', 'XXXX', 'ZIYY', 'ZIXX', 'IZZI', 'YYZI', 'XXZI', 'ZIZI', 'ZZII'],
              coeffs=[-6.51593168e-01+0.j,  2.97111485e-03+0.j,  2.66753022e-03+0.j,
  8.61457336e-02+0.j,  2.97111485e-03+0.j,  1.17505164e-01+0.j,
  7.45557377e-08+0.j,  5.60514952e-08+0.j,  7.45557377e-08+0.j,
  5.60514952e-08+0.j,  2.66753022e-03+0.j,  1.16687027e-01+0.j,
  7.45557377e-08+0.j,  5.60514952e-08+0.j,  7.45557377e-08+0.j,
  5.605149

## Task 2: Ground State Energy via VQE
This section sets up the Hartree-Fock reference state and the UCCSD ansatz, configures the statevector estimator with the COBYLA optimizer, and computes the ground-state electronic energy

In [ ]:
import time
import numpy as np
import matplotlib as plt

from qiskit.primitives import StatevectorEstimator
from qiskit.circuit.library import EfficientSU2
from qiskit_algorithms import VQE
from qiskit_algorithms.optimizers import COBYLA

from qiskit_nature.units import DistanceUnit
from qiskit_nature.second_q.drivers import PySCFDriver
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import JordanWignerMapper
from qiskit_nature.second_q.circuit.library import HartreeFock, UCCSD
from qiskit_nature.second_q.algorithms import GroundStateEigensolver, QEOM

### REQUIRED INPUT (Task 1's output): the active-space qubit-mapped problem


In [ ]:
HARTREE_TO_EV = 27.211386245988
MAPPER = JordanWignerMapper()



def build_problem(xyz: str, n_e: int = 2, n_orb: int = 2):
    driver = PySCFDriver(atom=xyz.strip(), basis="sto3g", charge=0, spin=0,
                          unit=DistanceUnit.ANGSTROM)
    full_problem = driver.run()
    transformer = ActiveSpaceTransformer(num_electrons=n_e, num_spatial_orbitals=n_orb)
    return transformer.transform(full_problem)

# ground-state VQE, UCCSD and hardware-efficient ansatz


In [ ]:
def run_vqe(problem, ansatz_type: str, maxiter: int = 300):
    num_spatial_orbitals = problem.num_spatial_orbitals
    num_particles = problem.num_particles
    hf_state = HartreeFock(num_spatial_orbitals, num_particles, MAPPER)

    if ansatz_type == "UCCSD":
        ansatz = UCCSD(num_spatial_orbitals, num_particles, MAPPER, initial_state=hf_state)
    elif ansatz_type == "hardware_efficient":
        num_qubits = 2 * num_spatial_orbitals
        ansatz = hf_state.compose(EfficientSU2(num_qubits, reps=2, entanglement="linear"))
    else:
        raise ValueError(ansatz_type)

    depth = ansatz.decompose(reps=3).depth()
    print(f"[{ansatz_type}] qubits={ansatz.num_qubits}  depth={depth}  "
          f"parameters={ansatz.num_parameters}")

    estimator = StatevectorEstimator()
    optimizer = COBYLA(maxiter=maxiter)

    n_evals = {"n": 0}
    vqe = VQE(estimator, ansatz, optimizer, callback=lambda *a: n_evals.__setitem__("n", n_evals["n"] + 1))
    vqe.initial_point = np.zeros(ansatz.num_parameters)

    solver = GroundStateEigensolver(MAPPER, vqe)
    t0 = time.time()
    result = solver.solve(problem)
    print(f"[{ansatz_type}] converged in {n_evals['n']} evaluations, {time.time() - t0:.1f}s")
    print(f"[{ansatz_type}] ground-state energy: {result.total_energies[0]:.6f} Ha")

    return result, solver


Task 3: Excited States via Quantum Equation of Motion ($q\text{EOM}$)This section utilizes the converged ground-state solver from Task 2 to construct the excitation operators in $q\text{EOM}$ and calculate vertical excitation energies ($\Delta E$)

In [ ]:
def run_qeom(problem, gse_solver, label: str):
    estimator = StatevectorEstimator()
    qeom = QEOM(gse_solver, estimator, excitations="sd")
    result = qeom.solve(problem)

    energies = result.total_energies
    ground = energies[0]
    print(f"[{label}] ground state: {ground:.6f} Ha")
    for i in range(1, len(energies)):
        gap_ev = (energies[i] - ground) * HARTREE_TO_EV
        print(f"[{label}] excited state {i}: {energies[i]:.6f} Ha  "
              f"(excitation energy: {gap_ev:.3f} eV)")
    return result



if __name__ == "__main__":
    EQUILIBRIUM_XYZ = "C 0 0 0.67; H 0 0.928 1.2355; H 0 -0.928 1.2355; " \
                       "C 0 0 -0.67; H 0 0.928 -1.2355; H 0 -0.928 -1.2355"

    problem = build_problem(EQUILIBRIUM_XYZ)  # Task 1's output, required by Tasks 2 & 3

    for ansatz_type in ["UCCSD", "hardware_efficient"]:
        print(f"\n{'=' * 60}\n{ansatz_type}\n{'=' * 60}")
        vqe_result, gse_solver = run_vqe(problem, ansatz_type)          # Task 2
        run_qeom(problem, gse_solver, ansatz_type)                       # Task 3


UCCSD
[UCCSD] qubits=4  depth=112  parameters=3
[UCCSD] converged in 78 evaluations, 3.1s
[UCCSD] ground-state energy: -77.116628 Ha
[UCCSD] ground state: -77.116628 Ha
[UCCSD] excited state 1: -76.941013 Ha  (excitation energy: 4.779 eV)
[UCCSD] excited state 2: -76.596800 Ha  (excitation energy: 14.145 eV)
[UCCSD] excited state 3: -76.407974 Ha  (excitation energy: 19.283 eV)

hardware_efficient
[hardware_efficient] qubits=4  depth=12  parameters=24


/tmp/ipykernel_2658/283416604.py:10: DeprecationWarning: The class ``qiskit.circuit.library.n_local.efficient_su2.EfficientSU2`` is deprecated as of Qiskit 2.1. It will be removed in Qiskit 3.0. Use the function qiskit.circuit.library.efficient_su2 instead.
  ansatz = hf_state.compose(EfficientSU2(num_qubits, reps=2, entanglement="linear"))


[hardware_efficient] converged in 300 evaluations, 2.6s
[hardware_efficient] ground-state energy: -77.116236 Ha
[hardware_efficient] ground state: -77.116236 Ha
[hardware_efficient] excited state 1: -76.942020 Ha  (excitation energy: 4.741 eV)
[hardware_efficient] excited state 2: -76.597830 Ha  (excitation energy: 14.107 eV)
[hardware_efficient] excited state 3: -76.408984 Ha  (excitation energy: 19.245 eV)


Task 5: Classical Active-Space Benchmark via CASCIThis section runs a Complete Active Space Configuration Interaction (CASCI) calculation using PySCF for both planar ($0^\circ$) and twisted ($90^\circ$) Ethylene ($\text{C}_2\text{H}_4$) geometries. By diagonalizing the Hamiltonian exactly within the $(2e, 2o)$ active space, it computes the ground-state electronic energy ($E_0$) and the first excited-state energy ($E_1$) to establish the active-space vertical excitation gap ($\Delta E$)

In [ ]:
from pyscf import gto, scf, mcscf

HARTREE_TO_EV = 27.211386


def get_ethylene_mol(twist_angle_deg=0.0):
    """
    Constructs PySCF Molecule object for Ethylene (C2H4) in STO-3G basis.
    Calculates 90-degree twist around the C=C bond axis.
    """
    rad = np.radians(twist_angle_deg)
    y_h = 0.92 * np.cos(rad)
    z_h = 0.92 * np.sin(rad)

    atom_str = f"""
    C  0.000000  0.000000   0.665000
    C  0.000000  0.000000  -0.665000
    H  0.000000  0.920000   1.230000
    H  0.000000 -0.920000   1.230000
    H  0.000000  {y_h:.6f}  {-1.230000 + z_h:.6f}
    H  0.000000 -{y_h:.6f}  {-1.230000 - z_h:.6f}
    """

    mol = gto.M(
        atom=atom_str.strip(),
        basis="sto3g",
        spin=0,
        charge=0,
        verbose=0,
    )
    return mol


def run_classical_benchmarks(twist_angle_deg=0.0):
    mol = get_ethylene_mol(twist_angle_deg)

    # 1. Standard Mean-Field Hartree-Fock Calculation
    mf = scf.RHF(mol)
    mf.kernel()

    # Active space parameters: 2 electrons in 2 active spatial orbitals
    ncas = 2
    nelecas = 2

    # CASCI (Exact Diagonalization in Active Space)
    casci_obj = mcscf.CASCI(mf, ncas=ncas, nelecas=nelecas)

    # Solve for both Ground State and First Excited State
    casci_obj.fcisolver.nroots = 2
    e_casci_tot, _, _, _, _ = casci_obj.kernel()

    # Safely handle array vs scalar indexing
    if isinstance(e_casci_tot, (list, np.ndarray)):
        e0_casci = e_casci_tot[0]
        e1_casci = e_casci_tot[1]
    else:
        e0_casci = e_casci_tot
        e1_casci = e_casci_tot

    delta_e_casci_ev = (e1_casci - e0_casci) * HARTREE_TO_EV

    return {
        "CASCI": {"E0": e0_casci, "E1": e1_casci, "Delta_E_eV": delta_e_casci_ev}
    }


if __name__ == "__main__":
    print("                 TASK 5: CLASSICAL CASCI BENCHMARKING                     ")

    geometries = [("Planar (0°)", 0.0), ("Twisted (90°)", 90.0)]

    for label, angle in geometries:
        benchmarks = run_classical_benchmarks(angle)
        print(f"\n>>> GEOMETRY: {label}")
        print("-" * 65)
        print(f"{'Method':<12} | {'Ground State E0 (Ha)':<22} | {'Vertical ΔE (eV)':<18}")
        print("-" * 65)
        print(f"{'CASCI':<12} | {benchmarks['CASCI']['E0']:<22.6f} | {benchmarks['CASCI']['Delta_E_eV']:<18.2f}")


                 TASK 5: CLASSICAL CASCI BENCHMARKING                     

>>> GEOMETRY: Planar (0°)
-----------------------------------------------------------------
Method       | Ground State E0 (Ha)   | Vertical ΔE (eV)  
-----------------------------------------------------------------
CASCI        | -77.116608             | 4.91              

>>> GEOMETRY: Twisted (90°)
-----------------------------------------------------------------
Method       | Ground State E0 (Ha)   | Vertical ΔE (eV)  
-----------------------------------------------------------------
CASCI        | -74.062176             | 1.91              


## Task 4 : using a real hardware with error mitigation techniques

In [ ]:
! pip install qiskit_ibm_runtime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 68.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 412.6/412.6 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.8/120.8 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 234.2/234.2 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.6/76.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 541.5/541.5 kB 32.4 MB/s eta 0:00:00


In [ ]:
from qiskit_ibm_runtime import QiskitRuntimeService

QiskitRuntimeService.save_account(
    token='Hdak_Y3b_ZdDA_Q2Y1gcl7nQAon5y....', #hidden for privacy
    instance="...",
    overwrite=True
)

In [ ]:
# TASK 4 real hardware imports
from qiskit_ibm_runtime import QiskitRuntimeService, EstimatorV2 as Estimator
from qiskit.transpiler import generate_preset_pass_manager

In [ ]:
# get the bound UCCSD circuit + Hamiltonian to send to hardware
# (re-running run_vqe here just to get a clean UCCSD-only result object,
# since your Task 2 loop overwrote vqe_result/gse_solver with hardware_efficient's)
vqe_result_hw, gse_solver_hw = run_vqe(problem, "UCCSD")

raw = vqe_result_hw.raw_result
bound_circuit = raw.optimal_circuit.assign_parameters(raw.optimal_point)
hamiltonian = MAPPER.map(problem.second_q_ops()[0])
E_EXACT = vqe_result_hw.total_energies[0].real

print(f"Reference (noiseless) energy: {E_EXACT:.6f} Ha")
print(f"Ansatz qubits: {bound_circuit.num_qubits}")

[UCCSD] qubits=4  depth=112  parameters=3


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


[UCCSD] converged in 78 evaluations, 8.3s
[UCCSD] ground-state energy: -77.116628 Ha
Reference (noiseless) energy: -77.116628 Ha
Ansatz qubits: 4


In [ ]:
service= QiskitRuntimeService()

In [ ]:
backend = service.least_busy(operational=True, simulator=False)
print(f"Using backend: {backend.name}")

Using backend: ibm_marrakesh


In [ ]:
pm = generate_preset_pass_manager(
    backend=backend,
    optimization_level=3,
    translation_method="translator",   # forces the built-in translator, skips ibm_dynamic_circuits
)
isa_circuit = pm.run(bound_circuit)
isa_hamiltonian = hamiltonian.apply_layout(isa_circuit.layout)

In [ ]:
print(f"Transpiled depth: {isa_circuit.depth()}")
print(f"Transpiled gate counts: {isa_circuit.count_ops()}")

Transpiled depth: 139
Transpiled gate counts: OrderedDict({'sx': 69, 'rz': 68, 'cz': 43, 'x': 7})


In [ ]:
# run at resilience_level 0 (raw), 1 (readout mitigation / TREX),
# and 2 (ZNE + TREX + gate twirling) — all handled server-side on real hardware
hw_results = {}
for level, label in [(0, "raw"), (1, "readout-mitigated (TREX)"),
                      (2, "ZNE + TREX + twirling")]:
    estimator = Estimator(mode=backend)
    estimator.options.resilience_level = level
    job = estimator.run([(isa_circuit, isa_hamiltonian)])
    energy = float(job.result()[0].data.evs)
    hw_results[label] = energy
    print(f"resilience_level={level} ({label}): E = {energy:.6f} Ha   "
          f"(job id: {job.job_id()})")

resilience_level=0 (raw): E = -1.066468 Ha   (job id: dafpk0dnj4cs73agne70)
resilience_level=1 (readout-mitigated (TREX)): E = -1.085226 Ha   (job id: dafpk3m42tqs73b0a190)
resilience_level=2 (ZNE + TREX + twirling): E = -1.151081 Ha   (job id: dafpkatnj4cs73agnf10)


In [ ]:
# get the constant energy terms Qiskit Nature adds back after solving
# in the active space (nuclear repulsion + frozen/inactive core energy)
energy_shift = sum(problem.hamiltonian.constants.values())
print(f"Energy shift (nuclear repulsion + core): {energy_shift:.6f} Ha")

hw_results_corrected = {label: e + energy_shift for label, e in hw_results.items()}

print(f"\n{'Method':<30}{'Energy (Ha)':<16}{'Error vs exact (mHa)':<22}")
print(f"{'Exact (statevector)':<30}{E_EXACT:<16.6f}{'0.000':<22}")
for label, e in hw_results_corrected.items():
    err_mha = (e - E_EXACT) * 1000
    print(f"{label:<30}{e:<16.6f}{err_mha:<22.3f}")

Energy shift (nuclear repulsion + core): -75.919140 Ha

Method                        Energy (Ha)     Error vs exact (mHa)  
Exact (statevector)           -77.116628      0.000                 
raw                           -76.985608      131.020               
readout-mitigated (TREX)      -77.004366      112.262               
ZNE + TREX + twirling         -77.070221      46.407                


In [ ]:
# TASK 4 (extension): qEOM excited states under noise, local simulator
from qiskit_aer.primitives import EstimatorV2 as AerEstimator
from qiskit_aer.noise import NoiseModel
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

fake_backend = FakeSherbrooke()
noise_model = NoiseModel.from_backend(fake_backend)

# shot-noise-only estimator (isolates finite-sampling noise from device noise)
shots_only_estimator = AerEstimator(options={"default_precision": 1e-3})

# realistic device-noise estimator (gate + readout errors from a real IBM snapshot)
noisy_estimator = AerEstimator(
    options={
        "backend_options": {"noise_model": noise_model},
        "default_precision": 1e-3,
    }
)

In [ ]:
# reusable qEOM runner, parameterized by which estimator does the measuring
def run_qeom_with_estimator(problem, gse_solver, estimator, label):
    qeom = QEOM(gse_solver, estimator, excitations="sd")
    result = qeom.solve(problem)
    energies = result.total_energies
    ground = energies[0]
    print(f"[{label}] ground state: {ground:.6f} Ha")
    gaps = []
    for i in range(1, len(energies)):
        gap_ev = (energies[i] - ground) * HARTREE_TO_EV
        gaps.append(gap_ev)
        print(f"[{label}] excited state {i}: {energies[i]:.6f} Ha  "
              f"(excitation energy: {gap_ev:.3f} eV)")
    return energies, gaps

In [ ]:
# get a clean UCCSD ground state to build all three qEOM runs on
# (the noiseless-optimized state same fixed reference used for the hardware run)
vqe_result_q4, gse_solver_q4 = run_vqe(problem, "UCCSD")

from qiskit import transpile
from qiskit_aer import AerSimulator

# decompose UCCSD's custom evolution gates into a basis AerEstimator can execute
transpiled_ansatz = transpile(gse_solver_q4.solver.ansatz, AerSimulator(), optimization_level=1)
gse_solver_q4.solver.ansatz = transpiled_ansatz

print("\n=== Exact (noiseless statevector) ===")
e_exact, gaps_exact = run_qeom_with_estimator(problem, gse_solver_q4, StatevectorEstimator(), "exact")

print("\n=== Shot noise only (no device noise) ===")
e_shots, gaps_shots = run_qeom_with_estimator(problem, gse_solver_q4, shots_only_estimator, "shots-only")

print("\n=== Realistic device noise (FakeSherbrooke) ===")
e_noisy, gaps_noisy = run_qeom_with_estimator(problem, gse_solver_q4, noisy_estimator, "device-noise")

[UCCSD] qubits=4  depth=112  parameters=3


/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_dsolve/linsolve.py:606: SparseEfficiencyWarning: splu converted its input to CSC format
  return splu(A).solve
/usr/local/lib/python3.13/dist-packages/scipy/sparse/linalg/_matfuncs.py:707: SparseEfficiencyWarning: spsolve is more efficient when sparse b is in the CSC matrix format
  return spsolve(Q, P)


[UCCSD] converged in 78 evaluations, 3.2s
[UCCSD] ground-state energy: -77.116628 Ha

=== Exact (noiseless statevector) ===
[exact] ground state: -77.116628 Ha
[exact] excited state 1: -76.941013 Ha  (excitation energy: 4.779 eV)
[exact] excited state 2: -76.596800 Ha  (excitation energy: 14.145 eV)
[exact] excited state 3: -76.407974 Ha  (excitation energy: 19.283 eV)

=== Shot noise only (no device noise) ===
[shots-only] ground state: -77.116628 Ha
[shots-only] excited state 1: -76.941205 Ha  (excitation energy: 4.773 eV)
[shots-only] excited state 2: -76.601344 Ha  (excitation energy: 14.022 eV)
[shots-only] excited state 3: -76.408256 Ha  (excitation energy: 19.276 eV)

=== Realistic device noise (FakeSherbrooke) ===
[device-noise] ground state: -77.116628 Ha
[device-noise] excited state 1: -76.939614 Ha  (excitation energy: 4.817 eV)
[device-noise] excited state 2: -76.596763 Ha  (excitation energy: 14.146 eV)
[device-noise] excited state 3: -76.406437 Ha  (excitation energy: 19.

In [ ]:
# comparison table for the excitation energies (triplet + singlet)
print(f"{'Condition':<20}{'Triplet (eV)':<16}{'Singlet V (eV)':<16}")
for label, gaps in [("Exact", gaps_exact), ("Shot noise", gaps_shots), ("Device noise", gaps_noisy)]:
    t = gaps[0] if len(gaps) > 0 else float('nan')
    s = gaps[1] if len(gaps) > 1 else float('nan')
    print(f"{label:<20}{t:<16.3f}{s:<16.3f}")

Condition           Triplet (eV)    Singlet V (eV)  
Exact               4.779           14.145          
Shot noise          4.773           14.022          
Device noise        4.817           14.146          
